# CAR-T compact model — main experimental design

Model code reused from `CART_reformulation_fifo.ipynb` (compact plan formulation).
- **Baseline**: minimise total cost, with the FIFO rule (C14–C16) on manufacturing start days at each facility.
- **Patient-aware**: minimise total cost + λ_D · E_D, no FIFO (λ_U = 0).

Fixed data come from the Houston 4 / T_MFE 7 workbook (`CART_instance_30patients_cap4.xlsx`, read-only):
hospitals c1..c10, facilities m1 Philadelphia, m2 Houston, m3 Los Angeles, transport times and unit costs,
CVM, HC, T_LS, T_MFE, T_QC and the Liu transition rates. Only three things are overwritten per run:
facility lines (factor C), ND (factor A) and the patients (generated below). Horizon NT = last collection day + ND + 2.

| Factor | Levels |
|---|---|
| A. ND | 41, 52 |
| B. Demand N (collection days 1..10) | 30, 60 |
| C. Lines (Philadelphia, Houston, LA) | C1 = (10,10,10), C2 = (4,10,10), C3 = (4,4,10), C4 = (4,4,4) |
| D. Urgent share | 10%, 20%, 30% |
| E. λ_D (Patient-aware only) | 25,000 / 75,000 / 150,000 / 300,000 |

48 combinations of A–D × 10 replications = 480 instances; per instance 1 Baseline + 4 Patient-aware = 2,400 solves.

In [ ]:
import os, re, time, json, pickle, itertools
import numpy as np
import pandas as pd
import openpyxl
import pyomo.environ as pyo
from pyomo.contrib.appsi.solvers import Highs
from pyomo.contrib.appsi.base import TerminationCondition as TC

OUT_DIR = "results_doe_main"
CACHE_DIR = os.path.join(OUT_DIR, "_cache")            # one file per finished instance, lets an interrupted run resume
for sub in ["instances", "figures", "_cache"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
pd.set_option("display.width", 280, "display.max_columns", 50, "display.max_rows", 300)

BASE_WORKBOOK = "CART_instance_30patients_cap4.xlsx"   # Houston 4 / T_MFE 7
ND_LEVELS = [41, 52]
N_LEVELS = [30, 60]
LAYOUTS = {"C1": (10, 10, 10), "C2": (4, 10, 10), "C3": (4, 4, 10), "C4": (4, 4, 4)}   # (m1, m2, m3) lines
SHARES = [0.10, 0.20, 0.30]
LAMBDAS = [25_000.0, 75_000.0, 150_000.0, 300_000.0]
LAMBDA_U = 0.0
REPS = list(range(1, 11))
TIME_LIMIT = 120.0                                     # seconds per solve
N_WORKERS = int(os.environ.get("DOE_WORKERS", "4"))    # instances solved in parallel, 1 HiGHS thread each
REGION_SHARE = {"East": 0.60, "Central": 0.13}         # West = the rest
# demand weights per hospital (Wan et al. 2026, total orders per hospital)
DEMAND_WEIGHT = {"Boston": 24, "New York": 29, "Philadelphia": 14, "Baltimore": 23, "Durham": 12, "Chicago": 14,
                 "Houston": 14, "Denver": 30, "Los Angeles": 37, "Palo Alto": 28}
MAIN_LAMBDA = 150_000.0
BASE_CASE = dict(ND=41, N=30, layout="C3", share=0.30)
pnum = lambda p: int(p[1:])

## 1. Fixed data (Houston 4 / T_MFE 7 workbook, read-only, `data_only=True`)

In [ ]:
def load_base(path):
    wb = openpyxl.load_workbook(path, data_only=True)

    def table(sheet, first_row, ncols, prefix):
        out = []
        for r in wb[sheet].iter_rows(min_row=first_row, max_col=ncols, values_only=True):
            if not (isinstance(r[0], str) and re.fullmatch(prefix + r"\d+", r[0])):
                break
            out.append(r)
        return out

    settings = {str(lab): val for lab, val, *_ in wb["Settings"].iter_rows(min_row=4, max_col=3, values_only=True) if lab}
    get = lambda start: next(v for k, v in settings.items() if k.startswith(start))
    b = dict(T_LS=int(get("T_LS")), T_MFE=int(get("T_MFE")), T_QC=int(get("T_QC")), HC=float(get("HC")),
             workbook_ND=int(get("ND")), workbook_NT=int(get("Horizon NT")))
    fac = table("Facilities", 4, 5, "m")
    b["M"] = [r[0] for r in fac]
    b["M_CITY"] = {r[0]: r[1] for r in fac}
    b["CVM"] = {r[0]: float(r[4]) for r in fac}
    b["workbook_FCAP"] = {r[0]: int(r[3]) for r in fac}
    sites = table("Sites", 4, 5, "c")
    b["C"] = [r[0] for r in sites]
    b["CITY"] = {r[0]: r[1] for r in sites}
    b["REGION"] = {r[0]: r[4] for r in sites}
    b["PAIRED_H"] = {r[0]: r[3] for r in sites}
    out_rows = table("Transport_out", 4, 7, "c")
    ret_rows = table("Transport_return", 4, 7, "m")
    b["DIST1"] = {(r[0], r[1]): float(r[3]) for r in out_rows}        # km, site -> facility
    b["TT1"] = {(r[0], r[1], r[2]): int(r[5]) for r in out_rows}
    b["U1"] = {(r[0], r[1], r[2]): float(r[6]) for r in out_rows}
    b["TT2"] = {(r[0], r[1], r[2]): int(r[5]) for r in ret_rows}
    b["U2"] = {(r[0], r[1], r[2]): float(r[6]) for r in ret_rows}
    b["J"] = sorted({k[2] for k in b["TT1"]} | {k[2] for k in b["TT2"]})
    b["MODE"] = {"j1": "fast", "j2": "slow"}
    q = {str(r[0]).split(":")[0]: float(r[1]) for r in wb["Clinical"].iter_rows(min_row=4, max_row=6, max_col=2, values_only=True)}
    b["q"] = q
    Pm = np.array([[1 - q["q_RU"] - q["q_RD"], q["q_RU"], q["q_RD"]], [0.0, 1 - q["q_UD"], q["q_UD"]], [0.0, 0.0, 1.0]])
    tau_max = 10 + max(ND_LEVELS) + 2
    pi, Pk = {}, np.eye(3)
    for tau in range(tau_max + 1):
        for gi, g in enumerate("RU"):
            for si, s in enumerate("RUD"):
                pi[g, tau, s] = float(Pk[gi, si])
        Pk = Pk @ Pm
    b["PI"] = pi
    dev = 0.0
    for r in wb["Pi"].iter_rows(min_row=5, max_col=7, values_only=True):
        if not isinstance(r[0], (int, float)):
            break
        for (g, s), v in zip([("R", "R"), ("R", "U"), ("R", "D"), ("U", "R"), ("U", "U"), ("U", "D")], r[1:7]):
            dev = max(dev, abs(pi[g, int(r[0]), s] - v))
    assert dev <= 1e-6, f"pi does not match the Pi sheet ({dev})"
    # nearest facility by distance, and the shortest possible turnaround (any facility, fastest legs, no capacity)
    b["NEAREST"] = {c: min(b["M"], key=lambda m: b["DIST1"][c, m]) for c in b["C"]}
    b["MIN_TRT"] = {c: b["T_LS"] + b["T_MFE"] + b["T_QC"]
                       + min(b["TT1"][c, m, j] for m in b["M"] for j in b["J"])
                       + min(b["TT2"][m, b["PAIRED_H"][c], j] for m in b["M"] for j in b["J"]) for c in b["C"]}
    return b


BASE = load_base(BASE_WORKBOOK)
city = lambda c: BASE["CITY"][c].split(",")[0]
assert set(DEMAND_WEIGHT) == {city(c) for c in BASE["C"]}
print(f"T_LS={BASE['T_LS']}  T_MFE={BASE['T_MFE']}  T_QC={BASE['T_QC']}  HC={BASE['HC']}  CVM={BASE['CVM']}  q={BASE['q']}")
print(f"Overwritten per run: ND (workbook {BASE['workbook_ND']}), NT (workbook {BASE['workbook_NT']}), "
      f"lines (workbook {BASE['workbook_FCAP']}), patients")
print(pd.DataFrame({c: dict(city=BASE["CITY"][c], region=BASE["REGION"][c], weight=DEMAND_WEIGHT[city(c)],
                            nearest=f"{BASE['NEAREST'][c]} {BASE['M_CITY'][BASE['NEAREST'][c]]}",
                            km=BASE["DIST1"][c, BASE["NEAREST"][c]], min_TRT=BASE["MIN_TRT"][c]) for c in BASE["C"]}).T.to_string())

## 2. Instance generation (common random numbers)
For each demand level N and replication r, one `numpy.random.default_rng(1000*N + r)` stream, drawn in this order:
1. hospitals, region by region (East, Central, West), each patient with probability proportional to the demand weights;
2. collection days, uniform integers on 1..10;
3. a random permutation of the patients — the urgent patients are the first round(share·N) of it.

Patients are then numbered p1..pN by collection day (ties in generation order); the number only breaks FIFO ties.
The same patients are used for every ND, capacity layout and urgent share with that N and r.

In [ ]:
def region_counts(N):
    nE, nC = round(REGION_SHARE["East"] * N), round(REGION_SHARE["Central"] * N)
    return {"East": nE, "Central": nC, "West": N - nE - nC}


def generate(N, r):
    rng = np.random.default_rng(1000 * N + r)
    hosp, region = [], []
    for reg, n in region_counts(N).items():
        sites = [c for c in BASE["C"] if BASE["REGION"][c] == reg]
        w = np.array([DEMAND_WEIGHT[city(c)] for c in sites], float)
        hosp += list(rng.choice(sites, size=n, p=w / w.sum()))
        region += [reg] * n
    days = rng.integers(1, 11, size=N)
    perm = rng.permutation(N)                             # over the generated patients
    rank = np.empty(N, int)
    rank[perm] = np.arange(N)                             # position of each generated patient in the permutation
    order = sorted(range(N), key=lambda i: (days[i], i))  # numbering p1..pN
    rows = []
    for k, i in enumerate(order, start=1):
        rows.append({"p": f"p{k}", "collection day": int(days[i]), "site": hosp[i], "hospital": BASE["PAIRED_H"][hosp[i]],
                     "city": BASE["CITY"][hosp[i]], "region": region[i], "permutation rank": int(rank[i]) + 1,
                     **{f"urgent at {int(round(s * 100))}%": "U" if rank[i] < round(s * N) else "R" for s in SHARES}})
    return pd.DataFrame(rows)


INSTANCES = {(N, r): generate(N, r) for N in N_LEVELS for r in REPS}
for (N, r), df in INSTANCES.items():
    df.to_csv(os.path.join(OUT_DIR, "instances", f"patients_N{N}_r{r:02d}.csv"), index=False)
    cnt = df.region.value_counts()
    assert (cnt.get("East", 0), cnt.get("Central", 0), cnt.get("West", 0)) == tuple(region_counts(N).values())
    for s in SHARES:
        assert (df[f"urgent at {int(round(s * 100))}%"] == "U").sum() == round(s * N)
    assert set(df[df["urgent at 10%"] == "U"].p) <= set(df[df["urgent at 20%"] == "U"].p) <= set(df[df["urgent at 30%"] == "U"].p)
summary = pd.DataFrame([{"N": N, "r": r, "seed": 1000 * N + r, "East": (df.region == "East").sum(),
                         "Central": (df.region == "Central").sum(), "West": (df.region == "West").sum(),
                         "last collection day": df["collection day"].max(),
                         **{city(c): (df.site == c).sum() for c in BASE["C"]}} for (N, r), df in INSTANCES.items()])
print(summary.to_string(index=False))

## 3. Model (compact plan formulation, from `CART_reformulation_fifo.ipynb`)

In [ ]:
def make_data(ND, N, layout, share, r):
    pat = INSTANCES[N, r]
    gcol = f"urgent at {int(round(share * 100))}%"
    d = dict(BASE)
    d.update(ND=ND, NT=int(pat["collection day"].max()) + ND + 2, FCAP=dict(zip(BASE["M"], LAYOUTS[layout])),
             P=list(pat.p), A=dict(zip(pat.p, pat["collection day"])), SITE=dict(zip(pat.p, pat.site)),
             HOSP=dict(zip(pat.p, pat.hospital)), G=dict(zip(pat.p, pat[gcol])))
    d["ORDER"] = sorted(d["P"], key=lambda p: (d["A"][p], pnum(p)))     # FIFO order
    return d


def enumerate_plans(d):
    plans = {}
    for p in d["P"]:
        c, h, a, g = d["SITE"][p], d["HOSP"][p], d["A"][p], d["G"][p]
        for m, j1, j2 in itertools.product(d["M"], d["J"], d["J"]):
            tt1, tt2 = d["TT1"][c, m, j1], d["TT2"][m, h, j2]
            arrival = a + d["T_LS"] + tt1
            for t in range(arrival, d["NT"] + 1):
                delivery = t + d["T_MFE"] + d["T_QC"] + tt2
                if delivery - a > d["ND"] or delivery > d["NT"]:
                    break
                wait = t - arrival
                plans[p, m, j1, j2, t] = dict(
                    arrival=arrival, wait=wait, delivery=delivery, TRT=delivery - a,
                    cost=d["CVM"][m] + tt1 * d["U1"][c, m, j1] + tt2 * d["U2"][m, h, j2] + d["HC"] * wait,
                    riskD=d["PI"][g, delivery - a, "D"], riskU=d["PI"][g, delivery - a, "U"])
    return plans


def build_model(d, plans, fifo):
    mdl = pyo.ConcreteModel()
    mdl.K = pyo.Set(initialize=list(plans), dimen=5, ordered=True)
    mdl.X = pyo.Var(mdl.K, domain=pyo.Binary)
    by_p, by_mt = {}, {}
    for k in plans:
        by_p.setdefault(k[0], []).append(k)
        by_mt.setdefault((k[1], k[4]), []).append(k)
    mdl.R1 = pyo.Constraint(d["P"], rule=lambda mdl, p: sum(mdl.X[k] for k in by_p[p]) == 1)
    def r2(mdl, m, t):
        ks = [k for tt in range(max(1, t - d["T_MFE"] + 1), t + 1) for k in by_mt.get((m, tt), [])]
        return sum(mdl.X[k] for k in ks) <= d["FCAP"][m] if ks else pyo.Constraint.Skip
    mdl.R2 = pyo.Constraint(d["M"], range(1, d["NT"] + 1), rule=r2)
    mdl.FIFO = pyo.ConstraintList()
    if fifo:   # C14-C16: z and S are linear expressions in X
        z = {(p, m): sum(mdl.X[k] for k in by_p[p] if k[1] == m) for p in d["P"] for m in d["M"]}
        S = {p: sum(k[4] * mdl.X[k] for k in by_p[p]) for p in d["P"]}
        order = d["ORDER"]
        for m in d["M"]:
            for i, p in enumerate(order):
                for q in order[i + 1:]:
                    mdl.FIFO.add(S[p] <= S[q] + d["NT"] * (2 - z[p, m] - z[q, m]))
    mdl.cost = pyo.Expression(expr=sum(plans[k]["cost"] * mdl.X[k] for k in plans))
    mdl.ED = pyo.Expression(expr=sum(plans[k]["riskD"] * mdl.X[k] for k in plans))
    mdl.EU = pyo.Expression(expr=sum(plans[k]["riskU"] * mdl.X[k] for k in plans))
    mdl.lamD = pyo.Param(mutable=True, initialize=0.0)
    mdl.obj = pyo.Objective(expr=mdl.cost + (mdl.lamD * mdl.ED + LAMBDA_U * mdl.EU if not fifo else 0), sense=pyo.minimize)
    return mdl


def new_solver():
    opt = Highs()
    opt.config.time_limit = TIME_LIMIT
    opt.config.mip_gap = 0.0
    opt.config.load_solution = False
    opt.highs_options = {"threads": 1, "random_seed": 0, "mip_abs_gap": 1e-9}
    return opt


def solve(opt, mdl):
    t0 = time.time()
    res = opt.solve(mdl)
    secs = time.time() - t0
    tc, obj, bound = res.termination_condition, res.best_feasible_objective, res.best_objective_bound
    has_sol = obj is not None and tc in (TC.optimal, TC.maxTimeLimit, TC.interrupted, TC.maxIterations)
    if has_sol:
        res.solution_loader.load_vars()
    status = {TC.optimal: "optimal", TC.infeasible: "infeasible", TC.infeasibleOrUnbounded: "infeasible"}.get(
        tc, ("time limit, feasible" if has_sol else "time limit, no solution") if tc == TC.maxTimeLimit else str(tc.name))
    gap = abs(obj - bound) / max(abs(obj), 1e-10) if has_sol and bound is not None else np.nan
    return dict(status=status, has_solution=has_sol, objective=obj if has_sol else np.nan,
                best_bound=bound if bound is not None else np.nan, mip_gap=gap, solve_s=secs)

## 4. Run one instance: Baseline (FIFO) + Patient-aware at four λ_D, metrics and checks

In [ ]:
def chosen(d, plans, mdl):
    rows = []
    for k in plans:
        if pyo.value(mdl.X[k]) > 0.5:
            p, m, j1, j2, t = k
            a = plans[k]
            rows.append(dict(p=p, cls=d["G"][p], site=d["SITE"][p], hospital=d["HOSP"][p], facility=m, j1=j1, j2=j2,
                             collection=d["A"][p], arrival=a["arrival"], start=t, delivery=a["delivery"], TRT=a["TRT"],
                             W=a["wait"], cost=a["cost"], riskD=a["riskD"], riskU=a["riskU"]))
    return pd.DataFrame(rows)


def check_plan(d, plans, tab, mdl, fifo):
    out = {}
    keys = [(r.p, r.facility, r.j1, r.j2, int(r.start)) for r in tab.itertuples()]
    out["one plan per patient"] = sorted(tab.p, key=pnum) == sorted(d["P"], key=pnum)
    tt1 = np.array([d["TT1"][r.site, r.facility, r.j1] for r in tab.itertuples()])
    tt2 = np.array([d["TT2"][r.facility, r.hospital, r.j2] for r in tab.itertuples()])
    arrival = tab.collection.values + d["T_LS"] + tt1
    delivery = tab.start.values + d["T_MFE"] + d["T_QC"] + tt2
    out["every plan admissible (start >= arrival, TRT <= ND, delivery <= NT)"] = bool(
        all(k in plans for k in keys) and (tab.start.values >= arrival).all() and (delivery - tab.collection.values <= d["ND"]).all()
        and (delivery <= d["NT"]).all() and (tab.hospital.values == tab.site.map(d["PAIRED_H"]).values).all())
    out["capacity respected for every m and t"] = all(
        ((tab.facility == m) & (tab.start >= t - d["T_MFE"] + 1) & (tab.start <= t)).sum() <= d["FCAP"][m]
        for m in d["M"] for t in range(1, d["NT"] + 1))
    out["TRT <= ND for every patient"] = bool((delivery - tab.collection.values <= d["ND"]).all())
    ed = sum(d["PI"][g, int(dl - a), "D"] for g, dl, a in zip(tab.cls, delivery, tab.collection))
    out["E_D recomputed from the plans = solver value (1e-6)"] = abs(ed - pyo.value(mdl.ED)) <= 1e-6
    if fifo:
        rank = {p: i for i, p in enumerate(d["ORDER"])}
        ok = True
        for m in d["M"]:
            s = tab[tab.facility == m].assign(o=lambda x: x.p.map(rank)).sort_values("o").start.values
            ok &= bool((np.diff(s) >= 0).all())
        out["FIFO holds in the Baseline solution"] = ok
    return out


def run_instance(key):
    ND, N, layout, share, r = key
    name = f"ND{ND}_N{N}_{layout}_U{int(round(share * 100))}_r{r:02d}"
    cache = os.path.join(CACHE_DIR, name + ".pkl")
    if os.path.exists(cache):
        return pickle.load(open(cache, "rb"))
    d = make_data(ND, N, layout, share, r)
    plans = enumerate_plans(d)
    pat = INSTANCES[N, r]
    n_phl = sum(BASE["NEAREST"][c] == "m1" for c in pat.site)
    common = dict(instance=name, ND=ND, N=N, layout=layout, lines=str(LAYOUTS[layout]), urgent_share=share, rep=r,
                  NT=d["NT"], n_urgent=sum(g == "U" for g in d["G"].values()), n_plans=len(plans),
                  rho_Philadelphia=n_phl * d["T_MFE"] / (d["FCAP"]["m1"] * 10),
                  E_D_floor=sum(d["PI"][d["G"][p], BASE["MIN_TRT"][d["SITE"][p]], "D"] for p in d["P"]))
    rows, plan_rows, checks = [], [], []
    runs = [("Baseline", 0.0)] + [("Patient-aware", l) for l in LAMBDAS]
    t0 = time.time(); mdl_b = build_model(d, plans, fifo=True); build_b = time.time() - t0
    t0 = time.time(); mdl_a = build_model(d, plans, fifo=False); build_a = time.time() - t0
    opt_b, opt_a = new_solver(), new_solver()
    base_tab = None
    for model, lam in runs:
        mdl, opt = (mdl_b, opt_b) if model == "Baseline" else (mdl_a, opt_a)
        mdl.lamD.set_value(lam)
        s = solve(opt, mdl)
        row = dict(common, model=model, lambda_D=lam, status=s["status"], objective=s["objective"], best_bound=s["best_bound"],
                   mip_gap=s["mip_gap"], solve_s=s["solve_s"], build_s=build_b if model == "Baseline" else build_a,
                   n_fifo_constraints=len(mdl.FIFO))
        if s["has_solution"]:
            tab = chosen(d, plans, mdl)
            for chk, ok in check_plan(d, plans, tab, mdl, fifo=(model == "Baseline")).items():
                checks.append(dict(instance=name, model=model, lambda_D=lam, check=chk, ok=bool(ok)))
            row.update(E_D=pyo.value(mdl.ED), E_U=pyo.value(mdl.EU), total_cost=pyo.value(mdl.cost),
                       mean_wait_urgent=tab.W[tab.cls == "U"].mean(), mean_wait_regular=tab.W[tab.cls == "R"].mean(),
                       mean_wait=tab.W.mean(), max_wait=tab.W.max(), max_TRT=tab.TRT.max())
            if model == "Baseline":
                base_tab, base_row = tab, row
                row.update(rerouted=0)
            elif base_tab is not None:
                diff = base_row["E_D"] - row["E_D"]
                extra = row["total_cost"] - base_row["total_cost"]
                row.update(E_D_difference=diff, relative_reduction=diff / base_row["E_D"], extra_cost=extra,
                           extra_cost_pct=100 * extra / base_row["total_cost"],
                           cost_per_protected_patient=extra / diff if diff > 0 else np.nan,
                           rerouted=int((tab.set_index("p").facility != base_tab.set_index("p").facility.reindex(tab.p)).sum()))
                # the Baseline plan is feasible for the Patient-aware model
                obj_on_base = base_row["total_cost"] + lam * base_row["E_D"]
                checks.append(dict(instance=name, model=model, lambda_D=lam,
                                   check="Patient-aware objective <= Patient-aware objective of the Baseline plan",
                                   ok=bool(row["objective"] <= obj_on_base + 1e-6 * max(1.0, abs(obj_on_base)))))
            plan_rows.append(tab.assign(instance=name, model=model, lambda_D=lam,
                                        out_mode=tab.j1.map(BASE["MODE"]), return_mode=tab.j2.map(BASE["MODE"])).drop(columns=["j1", "j2"]))
        rows.append(row)
    out = dict(rows=rows, plans=pd.concat(plan_rows, ignore_index=True) if plan_rows else pd.DataFrame(), checks=checks)
    pickle.dump(out, open(cache, "wb"))
    return out

## 5. Run all 480 instances (2,400 solves)
Instances run in parallel worker processes (`N_WORKERS`, one HiGHS thread each). Every finished instance is cached in
`results_doe_main/_cache/`, so re-running this cell resumes where it stopped.

In [ ]:
KEYS = [(ND, N, layout, share, r) for ND in ND_LEVELS for N in N_LEVELS for layout in LAYOUTS for share in SHARES for r in REPS]
assert len(KEYS) == 480
t_start = time.time()
if N_WORKERS > 1:
    import multiprocessing as mp
    with mp.get_context("fork").Pool(N_WORKERS) as pool:
        RESULTS = []
        for i, res in enumerate(pool.imap(run_instance, KEYS, chunksize=1), start=1):
            RESULTS.append(res)
            if i % 40 == 0:
                print(f"{i}/{len(KEYS)} instances done  [{time.time() - t_start:,.0f}s]", flush=True)
else:
    RESULTS = [run_instance(k) for k in KEYS]
print(f"all {len(RESULTS)} instances done in {time.time() - t_start:,.0f}s (cached instances load instantly)")

ALL = pd.DataFrame([row for res in RESULTS for row in res["rows"]])
COLS = ["instance", "ND", "N", "layout", "lines", "urgent_share", "rep", "model", "lambda_D", "status", "objective", "best_bound",
        "mip_gap", "E_D", "E_U", "total_cost", "E_D_difference", "relative_reduction", "extra_cost", "extra_cost_pct",
        "cost_per_protected_patient", "mean_wait_urgent", "mean_wait_regular", "mean_wait", "max_wait", "max_TRT", "rerouted",
        "rho_Philadelphia", "E_D_floor", "solve_s", "build_s", "n_plans", "n_fifo_constraints", "NT", "n_urgent"]
ALL = ALL[COLS]
ALL.to_csv(os.path.join(OUT_DIR, "all_runs.csv"), index=False)
PLANS = pd.concat([res["plans"] for res in RESULTS], ignore_index=True)
PLANS.to_csv(os.path.join(OUT_DIR, "all_plans.csv.gz"), index=False, compression="gzip")
CHECKS = pd.DataFrame([c for res in RESULTS for c in res["checks"]])
print(f"all_runs.csv: {len(ALL)} rows;  all_plans.csv.gz: {len(PLANS)} patient plans")

### Solve status

In [ ]:
st = ALL.groupby(["model", "lambda_D", "status"]).size().unstack(fill_value=0)
print(st.to_string())
print("\nsolve time (s) by model and N:")
print(ALL.groupby(["model", "N"]).solve_s.describe()[["mean", "50%", "max"]].round(2).to_string())
not_opt = ALL[ALL.status != "optimal"]
print(f"\n{len(not_opt)} of {len(ALL)} solves not proven optimal")
if len(not_opt):
    print(not_opt[["instance", "model", "lambda_D", "status", "objective", "best_bound", "mip_gap", "solve_s"]].to_string(index=False))

## 6. Tables (λ_D = 150,000 unless stated; mean [min–max] over the 10 replications)
For a factor level, each replication contributes the average over the other factors' combinations; the table shows the
mean and the min–max of those 10 replication averages. Relative reduction and extra cost % are averaged per instance.

In [ ]:
B = ALL[ALL.model == "Baseline"].set_index("instance")
A150 = ALL[(ALL.model == "Patient-aware") & (ALL.lambda_D == MAIN_LAMBDA)].set_index("instance")
PAIR = pd.DataFrame({
    "ND": B.ND, "N": B.N, "layout": B.layout, "urgent_share": B.urgent_share, "rep": B.rep, "rho": B.rho_Philadelphia,
    "E_D Baseline": B.E_D, "E_D Patient-aware": A150.E_D, "relative reduction": A150.relative_reduction,
    "extra cost %": A150.extra_cost_pct, "extra cost": A150.extra_cost,
    "urgent wait Baseline": B.mean_wait_urgent, "urgent wait Patient-aware": A150.mean_wait_urgent,
    "regular wait Baseline": B.mean_wait_regular, "regular wait Patient-aware": A150.mean_wait_regular})
METRICS = ["E_D Baseline", "E_D Patient-aware", "relative reduction", "extra cost %", "urgent wait Baseline", "urgent wait Patient-aware"]
FMT = {"E_D Baseline": "{:.3f}", "E_D Patient-aware": "{:.3f}", "relative reduction": "{:.1%}", "extra cost %": "{:.2f}%",
       "urgent wait Baseline": "{:.2f}", "urgent wait Patient-aware": "{:.2f}", "total cost": "{:,.0f}", "E_D": "{:.3f}",
       "extra cost": "{:,.0f}", "cost per protected patient": "{:,.0f}"}


def rep_stats(df, group_cols, metrics):
    """Average over everything except group_cols and rep, then mean/min/max over replications."""
    per_rep = df.groupby(group_cols + ["rep"])[metrics].mean()
    g = per_rep.groupby(level=list(range(len(group_cols))))
    return g.mean(), g.min(), g.max(), g.count()


def fmt_cell(mean, lo, hi, f):
    if pd.isna(mean):
        return "–"
    return f"{f.format(mean)} [{f.format(lo)}–{f.format(hi)}]"


def pretty(mean, lo, hi, metrics):
    out = pd.DataFrame(index=mean.index)
    for c in metrics:
        out[c] = [fmt_cell(a, b, c_, FMT[c]) for a, b, c_ in zip(mean[c], lo[c], hi[c])]
    return out


def long_csv(mean, lo, hi, metrics):
    parts = {}
    for c in metrics:
        parts[(c, "mean")], parts[(c, "min")], parts[(c, "max")] = mean[c], lo[c], hi[c]
    out = pd.DataFrame(parts)
    out.columns = [f"{a} ({b})" for a, b in out.columns]
    return out


# Table 1: main effects
t1_csv, t1_txt = [], []
for factor, label in [("ND", "ND"), ("N", "Demand N"), ("layout", "Capacity layout"), ("urgent_share", "Urgent share")]:
    mean, lo, hi, cnt = rep_stats(PAIR, [factor], METRICS)
    idx = [f"{label} = {v if factor != 'urgent_share' else f'{int(round(v * 100))}%'}"
           + (f" {LAYOUTS[v]}" if factor == "layout" else "") for v in mean.index]
    a = long_csv(mean, lo, hi, METRICS); a.index = idx; t1_csv.append(a)
    b = pretty(mean, lo, hi, METRICS); b.index = idx; t1_txt.append(b)
T1 = pd.concat(t1_csv); T1.index.name = "factor level"
T1.to_csv(os.path.join(OUT_DIR, "table1_main_effects.csv"))
print("TABLE 1 — main effects (λ_D = 150,000); mean [min–max] over 10 replications")
print(pd.concat(t1_txt).to_string())

In [ ]:
# Table 2: relative reduction by capacity layout x (ND, demand), urgent share 30%
sub = PAIR[PAIR.urgent_share == 0.30]
mean, lo, hi, cnt = rep_stats(sub, ["layout", "ND", "N"], ["relative reduction"])
T2_csv = long_csv(mean, lo, hi, ["relative reduction"]).unstack(["ND", "N"])
T2_csv.columns = [f"ND {nd}, N {n} — {stat}" for stat, nd, n in T2_csv.columns]
T2_csv.to_csv(os.path.join(OUT_DIR, "table2_relative_reduction_by_layout.csv"))
T2 = pretty(mean, lo, hi, ["relative reduction"])["relative reduction"].unstack(["ND", "N"])
T2.columns = [f"ND {nd}, N {n}" for nd, n in T2.columns]
T2.index = [f"{c} {LAYOUTS[c]}" for c in T2.index]
print("TABLE 2 — relative reduction at λ_D = 150,000, urgent share 30%; mean [min–max] over 10 replications")
print(T2.to_string())

In [ ]:
# Table 3: λ effect in the base case
bc = ALL[(ALL.ND == BASE_CASE["ND"]) & (ALL.N == BASE_CASE["N"]) & (ALL.layout == BASE_CASE["layout"])
         & (np.isclose(ALL.urgent_share, BASE_CASE["share"]))]
t3_rows, t3_csv = [], []
for lam in [0.0] + LAMBDAS:
    s = bc[bc.lambda_D == lam].set_index("rep")
    vals = {"total cost": s.total_cost, "E_D": s.E_D, "relative reduction": s.relative_reduction, "extra cost": s.extra_cost,
            "cost per protected patient": s.cost_per_protected_patient}
    lab = "0 (Baseline, FIFO)" if lam == 0 else f"{lam:,.0f}"
    t3_rows.append({"λ_D": lab, **{k: (fmt_cell(v.mean(), v.min(), v.max(), FMT[k]) if v.notna().any() else "–") for k, v in vals.items()},
                    "replications with a defined cost per protected patient": int(s.cost_per_protected_patient.notna().sum()) if lam else "–"})
    t3_csv.append({"lambda_D": lam, **{f"{k} ({st})": getattr(v, st)() for k, v in vals.items() for st in ["mean", "min", "max"]},
                   "n reps cost per protected patient defined": int(s.cost_per_protected_patient.notna().sum())})
T3 = pd.DataFrame(t3_rows).set_index("λ_D")
pd.DataFrame(t3_csv).to_csv(os.path.join(OUT_DIR, "table3_lambda_effect_base_case.csv"), index=False)
print(f"TABLE 3 — λ_D effect, base case ND {BASE_CASE['ND']}, N {BASE_CASE['N']}, {BASE_CASE['layout']} "
      f"{LAYOUTS[BASE_CASE['layout']]}, urgent 30%; mean [min–max] over 10 replications")
print(T3.to_string())

In [ ]:
# Table 4 (appendix): all 48 combinations
mean, lo, hi, cnt = rep_stats(PAIR, ["ND", "N", "layout", "urgent_share"], METRICS)
T4_csv = long_csv(mean, lo, hi, METRICS)
T4_csv.to_csv(os.path.join(OUT_DIR, "table4_appendix_48_combinations.csv"))
T4 = pretty(mean, lo, hi, METRICS)
T4.index = [f"ND {nd} | N {n} | {c} | {int(round(u * 100))}%" for nd, n, c, u in T4.index]
print("TABLE 4 — all 48 combinations (λ_D = 150,000); mean [min–max] over 10 replications")
print(T4.to_string())

## 7. Figures

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
LAYOUT_COLOR = {"C1": "#2a78d6", "C2": "#eb6834", "C3": "#1baf7a", "C4": "#eda100"}
BASE_COLOR, AWARE_COLOR = "#eb6834", "#2a78d6"
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
                     "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10})
FIG = os.path.join(OUT_DIR, "figures")

# Figure 1: benefit map
fig, ax = plt.subplots(figsize=(8.6, 5.4))
jit = np.random.default_rng(0)
for layout in LAYOUTS:
    for nd, mk in zip(ND_LEVELS, ["o", "^"]):
        s = PAIR[(PAIR.layout == layout) & (PAIR.ND == nd)]
        x = s.rho + jit.uniform(-0.07, 0.07, len(s))
        ax.scatter(x, s["relative reduction"] * 100, s=22, marker=mk, color=LAYOUT_COLOR[layout], alpha=0.55,
                   edgecolors=SURF, linewidths=0.6, label=f"{layout} {LAYOUTS[layout]}, ND {nd}", zorder=3)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xlabel("ρ Philadelphia = patients nearest Philadelphia × T_MFE ÷ (Philadelphia lines × 10)   (points jittered ±0.07)")
ax.set_ylabel("relative reduction in E_D at λ_D = 150k (%)")
ax.set_title("Benefit map: one point per instance (480)", loc="left", fontsize=12, color=INK)
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper left")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig1_benefit_map.png"), dpi=200); plt.show()

# Figure 2: cost vs E_D, base case
fig, ax = plt.subplots(figsize=(8.6, 5.4))
g = bc.groupby("lambda_D")
m_, lo_, hi_ = g[["total_cost", "E_D"]].mean(), g[["total_cost", "E_D"]].min(), g[["total_cost", "E_D"]].max()
k = 1e3
ax.errorbar(m_.total_cost / k, m_.E_D, xerr=[(m_.total_cost - lo_.total_cost) / k, (hi_.total_cost - m_.total_cost) / k],
            yerr=[m_.E_D - lo_.E_D, hi_.E_D - m_.E_D], fmt="none", ecolor=GRID, elinewidth=1.4, capsize=3, zorder=1)
ax.plot(m_.total_cost.iloc[1:] / k, m_.E_D.iloc[1:], color=AWARE_COLOR, lw=2, marker="o", ms=7, mec=SURF, mew=1.5, zorder=3,
        label="Patient-aware (no FIFO)")
ax.plot([m_.total_cost.iloc[0] / k], [m_.E_D.iloc[0]], ls="none", marker="D", ms=8, color=BASE_COLOR, mec=SURF, mew=1.5,
        zorder=4, label="Baseline (min cost, FIFO)")
for lam, row in m_.iterrows():
    ax.annotate("Baseline" if lam == 0 else f"λ_D = {lam / 1000:,.0f}k", (row.total_cost / k, row.E_D), xytext=(8, 6),
                textcoords="offset points", fontsize=9, color=INK2)
ax.set_xlabel("total cost (thousand USD)"); ax.set_ylabel("E_D: expected patients ineligible at delivery")
ax.set_title(f"Cost vs E_D, base case ND {BASE_CASE['ND']}, N {BASE_CASE['N']}, {BASE_CASE['layout']}, urgent 30% "
             "(mean, bars = min–max over 10 replications)", loc="left", fontsize=11, color=INK)
ax.legend(frameon=False); ax.margins(x=0.15, y=0.15)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig2_cost_vs_ED_base_case.png"), dpi=200); plt.show()

# Figure 3: mean wait of urgent and regular patients by capacity layout
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6), sharey=True)
x, w = np.arange(len(LAYOUTS)), 0.36
for ax, cls in zip(axes, ["urgent", "regular"]):
    for i, (mdl_name, col) in enumerate([("Baseline", BASE_COLOR), ("Patient-aware", AWARE_COLOR)]):
        vals = [PAIR[PAIR.layout == c][f"{cls} wait {mdl_name}"].mean() for c in LAYOUTS]
        bars = ax.bar(x + (i - 0.5) * (w + 0.03), vals, w, color=col, label=mdl_name, zorder=3)
        for b_, v in zip(bars, vals):
            ax.annotate(f"{v:.2f}", (b_.get_x() + b_.get_width() / 2, v), xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=9, color=INK)
    ax.set_xticks(x, [f"{c}\n{LAYOUTS[c]}" for c in LAYOUTS]); ax.grid(axis="x", visible=False)
    ax.set_title(f"Mean wait of {cls} patients (days)", loc="left", fontsize=11, color=INK)
axes[0].set_ylabel("days (mean over all instances of the layout)"); axes[0].legend(frameon=False)
axes[0].margins(y=0.15)
fig.suptitle("Baseline (FIFO) vs Patient-aware (λ_D = 150k), by capacity layout", x=0.01, ha="left", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig3_wait_by_class_and_layout.png"), dpi=200); plt.show()

## 8. Checks

In [ ]:
CHECKS.to_csv(os.path.join(OUT_DIR, "checks.csv"), index=False)
summ_chk = CHECKS.groupby("check").ok.agg(PASS="sum", total="count")
summ_chk["FAIL"] = summ_chk.total - summ_chk.PASS
print(summ_chk[["PASS", "FAIL"]].to_string())
fails = CHECKS[~CHECKS.ok]
print(f"\n{len(fails)} failed checks out of {len(CHECKS)}")
if len(fails):
    print(fails.to_string(index=False))
neg = ALL[(ALL.model == "Patient-aware") & (ALL.extra_cost < -1e-6)]
print(f"\nPatient-aware runs with NEGATIVE extra cost (cheaper than the FIFO Baseline; expected, not an error): "
      f"{len(neg)} of {(ALL.model == 'Patient-aware').sum()}")
nonpos = ALL[(ALL.model == "Patient-aware") & (ALL.E_D_difference <= 0)]
print(f"Patient-aware runs with E_D difference <= 0 (cost per protected patient left blank): {len(nonpos)}")
print(f"\nOVERALL: {'PASS' if len(fails) == 0 else 'FAIL'}  ({len(CHECKS)} checks)")